In [ ]:
# Model 1 ResNet50V2: baseline + trial 1 sampai 5 (akun Juna). Trial 6 sampai 10 dikerjakan Fatih.
# Pengaturan Kaggle: Add Input "yonobengkel12/simplicity-224-letterbox", Accelerator GPU, Internet ON.
# Jalankan dengan Save Version > Save & Run All (Commit) supaya output tersimpan.

import sys
import json
import time
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow import keras
from sklearn.metrics import precision_score

# Setup: clone repo kelompok lalu import data_utils (tidak disalin, tidak diubah)
REPO_URL = "https://github.com/0Raja0-RA/simplicity-cnn-comparison.git"
if Path("/kaggle").is_dir():
    ROOT = Path("/kaggle/working/simplicity-cnn-comparison")
    if not ROOT.is_dir():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    OUT = Path("/kaggle/working")
else:
    ROOT = Path.cwd().parent
    OUT = Path.cwd()
sys.path.insert(0, str(ROOT))
from common import data_utils as du

SIDIK_README = "b3e0629783c5"
assert du.sidik_config()[:12] == SIDIK_README, "Sidik data_utils.py beda dengan README, tanya grup dulu"
du.cek_perangkat()

# Pengaturan (aturan README: maks 40 epoch per trial, EarlyStopping patience 8)
NAMA = "juna"
JALANKAN_BASELINE = True
TRIAL_DIJALANKAN = [1, 2, 3, 4, 5]
EPOCH_MAKS = 40
PATIENCE = 8

# 10 kombinasi hasil random search seed 42. Akun ini hanya menjalankan 1 sampai 5.
# lr: learning rate tahap 1 (tahap 2 memakai lr/10)
# fine_tune: jumlah lapisan terakhir ResNet50V2 yang dibuka di tahap 2 (0 = tanpa tahap 2)
TRIALS = {
    1:  {"learning_rate": 1e-3, "dropout": 0.3, "dense_units": 128, "fine_tune": 60, "batch_size": 16},
    2:  {"learning_rate": 1e-3, "dropout": 0.2, "dense_units": 128, "fine_tune": 0,  "batch_size": 16},
    3:  {"learning_rate": 5e-4, "dropout": 0.2, "dense_units": 256, "fine_tune": 60, "batch_size": 16},
    4:  {"learning_rate": 5e-4, "dropout": 0.2, "dense_units": 128, "fine_tune": 30, "batch_size": 16},
    5:  {"learning_rate": 5e-4, "dropout": 0.2, "dense_units": 0,   "fine_tune": 30, "batch_size": 32},
    6:  {"learning_rate": 1e-3, "dropout": 0.3, "dense_units": 256, "fine_tune": 60, "batch_size": 32},
    7:  {"learning_rate": 1e-3, "dropout": 0.3, "dense_units": 128, "fine_tune": 30, "batch_size": 16},
    8:  {"learning_rate": 1e-4, "dropout": 0.3, "dense_units": 256, "fine_tune": 0,  "batch_size": 32},
    9:  {"learning_rate": 1e-3, "dropout": 0.3, "dense_units": 0,   "fine_tune": 60, "batch_size": 16},
    10: {"learning_rate": 1e-4, "dropout": 0.5, "dense_units": 128, "fine_tune": 0,  "batch_size": 32},
}
BASELINE = {"learning_rate": 1e-3, "dropout": 0.0, "dense_units": 0, "fine_tune": 0, "batch_size": 32}


class Presisi(keras.callbacks.Callback):
    """Precision macro train dan validation tiap epoch, dihitung tanpa augmentasi."""

    def __init__(self, train_ds, val_ds):
        super().__init__()
        self.train_ds, self.val_ds = train_ds, val_ds
        self.precision, self.val_precision = [], []

    def _hitung(self, ds):
        asli, tebak = [], []
        for x, y in ds:
            tebak.append(np.argmax(self.model(x, training=False).numpy(), axis=1))
            asli.append(y.numpy())
        return precision_score(np.concatenate(asli), np.concatenate(tebak), average="macro", zero_division=0)

    def on_epoch_end(self, epoch, logs=None):
        self.precision.append(self._hitung(self.train_ds))
        self.val_precision.append(self._hitung(self.val_ds))


def buat_model(cfg):
    inputs = keras.Input(shape=du.IMG_SIZE + (3,))
    x = du.get_augmentation()(inputs)                      # aktif hanya saat training
    x = keras.applications.resnet_v2.preprocess_input(x)  # normalisasi bawaan ResNet50V2
    base = keras.applications.ResNet50V2(include_top=False, weights="imagenet", pooling="avg")
    base.trainable = False
    x = base(x, training=False)
    if cfg["dropout"] > 0:
        x = keras.layers.Dropout(cfg["dropout"])(x)
    if cfg["dense_units"] > 0:
        x = keras.layers.Dense(cfg["dense_units"], activation="relu")(x)
        if cfg["dropout"] > 0:
            x = keras.layers.Dropout(cfg["dropout"])(x)
    outputs = keras.layers.Dense(10, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="resnet50v2"), base


def kompilasi(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])


def fit_tahap(model, train_ds, val_ds, epochs, riwayat):
    cb = Presisi(train_ds, val_ds)
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)
    h = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=[cb, es], verbose=2)
    for k in ("loss", "val_loss", "accuracy", "val_accuracy"):
        riwayat[k] += [float(v) for v in h.history[k]]
    riwayat["precision"] += [float(v) for v in cb.precision]
    riwayat["val_precision"] += [float(v) for v in cb.val_precision]
    return len(h.history["loss"])


def latih(label, cfg):
    keras.backend.clear_session()
    du.set_seed(du.SEED)
    train_ds, val_ds = du.load_data(batch_size=cfg["batch_size"])
    model, base = buat_model(cfg)
    riwayat = {k: [] for k in ("loss", "val_loss", "accuracy", "val_accuracy", "precision", "val_precision")}
    mulai = time.time()

    # Tahap 1: backbone beku, hanya kepala klasifikasi yang dilatih
    kompilasi(model, cfg["learning_rate"])
    maks_tahap1 = EPOCH_MAKS if cfg["fine_tune"] == 0 else EPOCH_MAKS // 2
    epoch_tahap1 = fit_tahap(model, train_ds, val_ds, maks_tahap1, riwayat)
    epoch_tahap2 = 0

    # Tahap 2: buka sebagian lapisan terakhir backbone (BatchNorm tetap beku)
    if cfg["fine_tune"] > 0:
        bobot_tahap1 = model.get_weights()
        loss_tahap1 = model.evaluate(val_ds, verbose=0)[0]
        base.trainable = True
        for layer in base.layers[:-cfg["fine_tune"]]:
            layer.trainable = False
        for layer in base.layers:
            if isinstance(layer, keras.layers.BatchNormalization):
                layer.trainable = False
        kompilasi(model, cfg["learning_rate"] / 10)
        epoch_tahap2 = fit_tahap(model, train_ds, val_ds, EPOCH_MAKS - epoch_tahap1, riwayat)
        if model.evaluate(val_ds, verbose=0)[0] > loss_tahap1:
            model.set_weights(bobot_tahap1)
            print("Tahap 2 tidak menurunkan val_loss, bobot tahap 1 dipakai.")

    waktu = time.time() - mulai
    val_loss, val_acc = model.evaluate(val_ds, verbose=0)
    train_loss, train_acc = model.evaluate(train_ds, verbose=0)
    tebak = np.argmax(model.predict(val_ds, verbose=0), axis=1)
    val_prec = precision_score(du.ambil_label(val_ds), tebak, average="macro", zero_division=0)

    model.save(OUT / f"resnet50v2_{label}.keras")   # tetap di output Kaggle, jangan di-commit
    with open(OUT / f"riwayat_{label}.json", "w") as f:
        json.dump({"epoch_tahap1": epoch_tahap1, **riwayat}, f)

    # kolom sesuai README: trial, parameter, val_acc, val_loss, best_epoch, train_time_sec
    baris = {"trial": label, **cfg, "val_acc": round(val_acc, 4), "val_loss": round(val_loss, 4),
             "best_epoch": int(np.argmin(riwayat["val_loss"])) + 1, "train_time_sec": round(waktu, 1)}
    detail = {"trial": label, "epoch_tahap1": epoch_tahap1, "epoch_tahap2": epoch_tahap2,
              "train_acc": round(train_acc, 4), "train_loss": round(train_loss, 4),
              "val_precision": round(val_prec, 4)}
    print(baris)
    return baris, detail, riwayat, epoch_tahap1


def plot_riwayat(riwayat, judul, epoch_tahap1, simpan):
    fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
    for a, (k, nama) in zip(ax, [("loss", "Loss"), ("accuracy", "Accuracy"), ("precision", "Precision (macro)")]):
        a.plot(riwayat[k], label="train")
        a.plot(riwayat["val_" + k], label="validation")
        if epoch_tahap1 < len(riwayat[k]):
            a.axvline(epoch_tahap1 - 0.5, color="gray", linestyle=":", label="mulai fine tune")
        a.set_title(f"{judul} {nama}")
        a.set_xlabel("epoch")
        a.legend()
    plt.tight_layout()
    plt.savefig(simpan, dpi=120)
    plt.show()


# Jalankan baseline lalu trial milik akun ini
tabel, detail_semua, grafik = [], [], []

if JALANKAN_BASELINE:
    print("\nBaseline:", BASELINE)
    baris, detail, riw, e1 = latih("baseline", BASELINE)
    pd.DataFrame([baris]).to_csv(OUT / "baseline_resnet50v2.csv", index=False)
    pd.DataFrame([detail]).to_csv(OUT / "detail_baseline.csv", index=False)
    grafik.append(("baseline", riw, e1))

for n in TRIAL_DIJALANKAN:
    print(f"\nTrial {n}:", TRIALS[n])
    baris, detail, riw, e1 = latih(f"trial{n}", TRIALS[n])
    baris["trial"], detail["trial"] = n, n
    tabel.append(baris)
    detail_semua.append(detail)
    grafik.append((f"trial{n}", riw, e1))
    pd.DataFrame(tabel).to_csv(OUT / f"trials_{NAMA}.csv", index=False)       # disimpan tiap trial selesai
    pd.DataFrame(detail_semua).to_csv(OUT / f"detail_{NAMA}.csv", index=False)

for label, riw, e1 in grafik:
    plot_riwayat(riw, f"ResNet50V2 {label}", e1, OUT / f"history_resnet50v2_{label}.png")

print("\nRingkasan trial", NAMA)
print(pd.DataFrame(tabel).to_string(index=False))
